# Dazo v0 — ProofWriter recurrent-depth pilot

This notebook clones `8dazo/dazo`, validates the architecture, trains the first shallow-depth ProofWriter pilot on a free Colab GPU, and measures the same checkpoint at 1/2/4/6/8 recurrent steps.

**Runtime:** choose a GPU runtime (T4 is enough for the pilot).


In [ ]:
!nvidia-smi || true
!git clone -q https://github.com/8dazo/dazo.git
%cd dazo
!pip -q install -e ".[train]"


## 1. Architecture invariants


In [ ]:
!pytest -q tests/test_core.py


## 2. Stream a small ProofWriter pilot

Training keeps only examples with gold query depth ≤3. Validation/test retain deeper cases, which is essential for the compute-scaling test.


In [ ]:
!python scripts/prepare_proofwriter.py \
  --output data/proofwriter-pilot \
  --train-max-depth 3 \
  --limit-train 3000 \
  --limit-eval 1000


## 3. Train the recurrent head

The mmBERT-small backbone is frozen in Dazo-0. Training randomly samples recurrent budgets from 1/2/3/4/6/8 per batch instead of always training at one fixed depth.


In [ ]:
!python train.py \
  --config configs/dazo-v0-small.json \
  --train data/proofwriter-pilot/train.jsonl \
  --eval data/proofwriter-pilot/validation.jsonl \
  --output outputs/dazo-proofwriter-pilot \
  --epochs 1 \
  --batch-size 4 \
  --grad-accum 2 \
  --lr 2e-4 \
  --depth-budgets 1,2,3,4,6,8


## 4. Test-time compute scaling

This is the main Dazo-0 question: does the *same checkpoint* improve on deeper examples when we give it more recurrent inference steps?


In [ ]:
!python evaluate.py \
  --model outputs/dazo-proofwriter-pilot/final \
  --data data/proofwriter-pilot/test.jsonl \
  --batch-size 8 \
  --loops 1,2,4,6,8


## 5. Full run after the pilot

If the pipeline is stable, increase to ~20k–50k shallow-depth examples and 2–3 epochs before interpreting the recurrence curve. Do **not** claim reasoning improvement from this small smoke run alone.


In [ ]:
# Example full-data preparation:
# !python scripts/prepare_proofwriter.py --output data/proofwriter --train-max-depth 3 --limit-train 30000 --limit-eval 5000
#
# Then train to outputs/dazo-proofwriter and evaluate with the same 1,2,4,6,8 loop sweep.
